In [1]:
# ── Imports & Async Patch ─────────────────────────────────────
import nest_asyncio
nest_asyncio.apply()   # Lets us use asyncio.run() inside Jupyter

import os
import json
import asyncio
import httpx
from openai import OpenAI
from pprint import pprint

In [2]:
# !pip install openai python-dotenv pandas
import pandas as pd
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI
import textwrap

import truststore
truststore.inject_into_ssl()



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

MODEL = 'gpt-5-nano'




client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

API key loaded successfully.
OpenAI client ready.


# World without MCP

In [4]:
# ══════════════════════════════════════════════════════════════
# STEP 1: Write the actual function that calls the external API
# ══════════════════════════════════════════════════════════════

def get_weather_manual(latitude: float, longitude: float) -> dict:
    """Call the Open-Meteo API to get current weather for given coordinates."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }
    response = httpx.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    weather = data["current_weather"]
    return {
        "temperature_celsius": weather["temperature"],
        "windspeed_kmh": weather["windspeed"],
        "weather_code": weather["weathercode"],
        "is_day": weather["is_day"] == 1
    }


import httpx

#def geocode_city(city: str):
#    url = "https://nominatim.openstreetmap.org/search"
#    params = {"q": city, "format": "json", "limit": 1}
#    headers = {"User-Agent": "mcp-workshop-demo/1.0"}
#    r = httpx.get(url, params=params, headers=headers, timeout=20)
#    r.raise_for_status()
#    data = r.json()
#    if not data:
#        return None
#    return {"latitude": float(data[0]["lat"]), "longitude": float(data[0]["lon"])}

#print(geocode_city("Tokyo"))

# Quick test
result = get_weather_manual(48.8566, 2.3522)  # Paris
print("Direct API call result for Paris:")
pprint(result)

Direct API call result for Paris:
{'is_day': True,
 'temperature_celsius': 17.3,
 'weather_code': 0,
 'windspeed_kmh': 10.2}


In [5]:

weather_tool_schema = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current weather for a location given its latitude and longitude. "
                   "Returns temperature in Celsius, wind speed, and weather conditions.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {
                "type": "number",
                "description": "Latitude of the location (e.g., 48.8566 for Paris)"
            },
            "longitude": {
                "type": "number",
                "description": "Longitude of the location (e.g., 2.3522 for Paris)"
            }
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}

print("Tool schema defined ✅")
print(json.dumps(weather_tool_schema, indent=2))

Tool schema defined ✅
{
  "type": "function",
  "name": "get_weather",
  "description": "Get the current weather for a location given its latitude and longitude. Returns temperature in Celsius, wind speed, and weather conditions.",
  "parameters": {
    "type": "object",
    "properties": {
      "latitude": {
        "type": "number",
        "description": "Latitude of the location (e.g., 48.8566 for Paris)"
      },
      "longitude": {
        "type": "number",
        "description": "Longitude of the location (e.g., 2.3522 for Paris)"
      }
    },
    "required": [
      "latitude",
      "longitude"
    ],
    "additionalProperties": false
  },
  "strict": true
}


In [6]:

def chat_with_weather_tool(user_message: str) -> str:
    """
    A complete manual tool-use loop with the Responses API:
    1. Send user message + tool definitions
    2. Check if the model wants to call a function
    3. Execute the function locally
    4. Send the result back
    5. Return the final answer
    """

    # ── First call: the model may request a tool call ──
    response = client.responses.create(
        model=MODEL,
        instructions="You are a helpful assistant. Use the get_weather tool when asked about weather.",
        input=user_message,
        tools=[weather_tool_schema],
    )

    while True:
        function_calls = [item for item in response.output if item.type == "function_call"]

        if not function_calls:
            return response.output_text

        tool_outputs = []

        for fc in function_calls:
            arguments = json.loads(fc.arguments)

            print(f"🔧 LLM requested tool: {fc.name}")
            print(f"   Arguments: {arguments}")

            if fc.name == "get_weather":
                tool_result = get_weather_manual(
                    latitude=arguments["latitude"],
                    longitude=arguments["longitude"],
                )
            else:
                tool_result = {"error": f"Unknown tool: {fc.name}"}

            print(f"   Result: {tool_result}")

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": fc.call_id,
                "output": json.dumps(tool_result),
            })

        response = client.responses.create(
            model=MODEL,
            instructions="You are a deadpool, but have been hired as assistant.",
            previous_response_id=response.id,   # key fix
            input=tool_outputs,                 # only tool outputs
            tools=[weather_tool_schema],        # keep tools available for another round
        )
        return response.output_text

    else:
        return response.output_text

# Let's test it!
answer = chat_with_weather_tool("What's the weather like in Tokyo right now?")
print("\n" + "="*60)
print("💬 Final Answer:")
print(answer)

🔧 LLM requested tool: get_weather
   Arguments: {'latitude': 35.6895, 'longitude': 139.6917}
   Result: {'temperature_celsius': 8.4, 'windspeed_kmh': 3.2, 'weather_code': 3, 'is_day': False}

💬 Final Answer:
Tokyo right now is about 8.4°C with a light breeze (3.2 km/h). It’s nighttime out there. The weather code it gave me is 3, which is a bit of a vague nerdy label—want me to translate that into a plain-English description (e.g., cloudy, partly cloudy)? If you’re stepping out, it’s a good coat night.


# Steps we've to follow

1. You define functions to perform specific tasks.
2. You define json-schema for providing tool instructions to the model.
3. You call the model with the user query and the tool instructions.
4. The model generates a response based on the user query and the tool instructions.
5. You check for tool calls in the model's response.
6. If there are tool calls, you execute the corresponding functions and get the results.
7. You provide the results back to the model for further processing.
8. The model generates a final response based on the user query, tool instructions, and the

# Problems with it
1. if function definition changes, I'm the one who has to update it, and change the json schema as well.
2. I need to design the custom integration for every model. 




tool_list[
    {'function_name': 'get_current_weather', 'description': 'Get the current weather in a given location', 'parameters': {'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'The city and state, e.g. San Francisco, CA'}, 'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']}}}}

    {'function_name': 'get_forecast', 'description': 'Get the weather forecast for a given location', 'parameters': {'type': 'object', 'properties': {'location': {'type': 'string', 'description': 'The city and state, e.g. San Francisco, CA'}, 'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']}}}}
]


## 2.3 How MCP Communication Works

There are really **two phases**: **tool discovery** and **tool execution**.

### Phase 1: Tool Discovery

Before the LLM can decide to call something like `get_weather`, the host application must first learn what tools the MCP server provides.

```
Host application starts / connects to MCP server
           │
           ▼
┌─────────────────────┐
│   HOST APPLICATION   │
│                      │
│  ┌────────────────┐  │  1. Host initializes MCP connection
│  │   MCP Client   │──┼──── 2. Client requests available tools ────────┐
│  └────────────────┘  │                                                 │
│                      │                                                 ▼
│                      │                                    ┌────────────────────┐
│                      │                                    │    MCP Server      │
│                      │                                    │   (Weather)        │
│                      │                                    │                    │
│                      │◄──── 3. Server returns tool list ──│  Tools:            │
│                      │                                    │  - get_weather     │
│                      │                                    │  - get_forecast    │
│                      │                                    └────────────────────┘
│                      │
│  ┌────────────────┐  │  4. Host converts MCP tool schemas
│  │   LLM Engine   │◄─┼──── into the LLM's tool/function format
│  └────────────────┘  │
└─────────────────────┘
```

At this point, the LLM knows that a tool named `get_weather` exists, what it does, and what arguments it expects.

### Phase 2: Tool Execution

Now when the user asks a question, the LLM can choose one of the discovered tools.

```
User: "What's the weather in Tokyo?"
           │
           ▼
┌─────────────────────┐
│   HOST APPLICATION   │  1. User sends message
│                      │  2. Host forwards message + available tool definitions to LLM
│  ┌────────────────┐  │  3. LLM says: "Call get_weather(location='Tokyo')"
│  │   LLM Engine   │  │
│  └───────┬────────┘  │
│          │           │  4. Host tells MCP Client to invoke that tool
│  ┌───────▼────────┐  │
│  │   MCP Client   │──┼──── 5. Client sends tool request to server ────┐
│  └───────▲────────┘  │                                                 │
│          │           │                                                 ▼
│          │           │                                    ┌────────────────────┐
│          │           │                                    │    MCP Server      │
│          │           │                                    │   (Weather)        │
│          │           │                                    │                    │
│          │           │  6. Server calls weather API       │                    │
│          │           │                                    └────────────────────┘
│  ┌───────┴────────┐  │
│  │   LLM Engine   │◄─┼──── 7. Client returns tool result to host
│  │  (with result) │  │      and host passes it back to LLM
│  └───────┬────────┘  │
│          │           │  8. LLM generates natural-language answer
└──────────┼───────────┘
           ▼
User: "It's 22°C and sunny in Tokyo right now!"
```

The key insight: **the LLM never talks directly to the MCP server.** The host application orchestrates everything. The MCP client handles protocol details like discovery, serialization, transport, and error handling. The LLM only sees the tool definitions the host gives it, and later the tool results the host sends back.

## 2.4 What Does MCP Give Us?

| Without MCP                                                          | With MCP                                                           |
| -------------------------------------------------------------------- | ------------------------------------------------------------------ |
| Custom JSON schema per tool, per app                                 | Server defines tools once; clients can discover them automatically |
| Manual function routing in each app                                  | Protocol standardizes discovery and invocation                     |
| Tool definitions hardcoded into hosts                                | Host can fetch tools dynamically from servers                      |
| Auth logic scattered everywhere                                      | Server manages its own auth                                        |
| Schema drift between implementation and exposed function definitions | Tool schemas come directly from the server's definitions           |
| N tools × M apps = N×M integrations                                  | N tools + M apps = N+M integrations                                |

## Important Clarification

What actually happens is:

1. The **host** uses the **MCP client** to ask the server for its tools.
2. The **server** returns tool names, descriptions, and schemas.
3. The **host** passes those tool definitions to the **LLM**.
4. The **LLM** decides which tool to call based on that information.

In [7]:
# ══════════════════════════════════════════════════════════════
# OpenAI Agents SDK + MCP — The Cleanest Approach
# ══════════════════════════════════════════════════════════════

from agents import Agent, Runner, enable_verbose_stdout_logging
from agents.mcp import MCPServerStdio
#enable_verbose_stdout_logging()

async def agents_sdk_demo():
    """
    Using OpenAI Agents SDK with our MCP server.
    Notice how we don't write ANY tool routing code!
    """

    mcp_server = MCPServerStdio(
        params={
            "command": "python",
            "args": ["weather_server.py"]
        }
    )

    async with mcp_server:
        agent = Agent(
            name="Weather Assistant",
            instructions=(
                "You are a helpful weather assistant. "
                "Use the available tools to answer questions about weather. "
                "Always provide temperatures in both Celsius and Fahrenheit."
            ),
            mcp_servers=[mcp_server],
        )

        print("🤖 Running agent with query...\n")
        result = await Runner.run(
            starting_agent=agent,
            input="What's the weather in New York City right now? "
                  "Also, what do the weather codes mean?"
        )

        print("💬 Agent's Response:")
        print(result.final_output)


asyncio.run(agents_sdk_demo())

🤖 Running agent with query...

💬 Agent's Response:
Right now in New York City, the temperature is 1.8°C (35.2°F) with wind speeds of 29.0 km/h. The weather code is 1.

Weather codes are used to quickly describe the current weather conditions, such as clear sky, cloudy, rain, or snow. If you'd like, I can explain what weather code 1 stands for or provide a general overview of common weather codes. Would you like more details?


# Two MCP servers

In [8]:
# ══════════════════════════════════════════════════════════════
# Multi-Server Agent — Weather + Notes
# ══════════════════════════════════════════════════════════════

async def multi_server_demo():
    weather_server = MCPServerStdio(
        params={"command": "python", "args": ["weather_server.py"]}
    )
    notes_server = MCPServerStdio(
        params={"command": "python", "args": ["notes_server.py"]}
    )

    async with weather_server, notes_server:
        agent = Agent(
            name="Personal Assistant",
            instructions=(
                "You are a personal assistant with access to weather data and a notes system. "
                "You can check weather anywhere in the world and manage notes. "
                "When the user asks about weather, check it and offer to save the info as a note."
            ),
            mcp_servers=[weather_server, notes_server],
        )

        print("🚀 Agent has tools from BOTH servers!\n")

        result = await Runner.run(
            starting_agent=agent,
            input=(
                "Check the weather in Tokyo and Paris right now, "
                "then save a note with the comparison results. "
                "Title it 'Weather Check' with tags: weather, travel"
            )
        )

        print("💬 Agent's Response:")
        print(result.final_output)

asyncio.run(multi_server_demo())

🚀 Agent has tools from BOTH servers!

💬 Agent's Response:
Here is the current weather comparison:

- Tokyo: 8.1°C, wind speed 3.7 km/h
- Paris: 17.2°C, wind speed 10.9 km/h

Paris is currently both warmer and windier than Tokyo.

I have saved a note titled "Weather Check" with the tags "weather, travel" containing these results. If you’d like to review, edit, or add more details, just let me know!
